# 📊 Notebook 01 — Exploratory Data Analysis
**SME Retail Revenue Maximization | Inventory-Demand Co-optimization**

> เป้าหมาย: เข้าใจ data ทุกตาราง, data quality, และหา business insight เบื้องต้น

## 0. Setup & Data Load

In [ ]:
# Install dependencies (Colab)
# !pip install pandas numpy matplotlib seaborn plotly openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')
print("✅ Libraries loaded")

In [ ]:
# ── Load all tables ──────────────────────────────────────────
# Colab: upload files หรือ mount Google Drive แล้วเปลี่ยน path
DATA_PATH = "data/raw/"   # เปลี่ยนตาม path จริง

sales    = pd.read_csv(DATA_PATH + "sales_transaction.csv",  parse_dates=["datetime"])
po       = pd.read_csv(DATA_PATH + "purchasing_order.csv",   parse_dates=["po_date","arrival_date","expire_date","manufacturing_date"])
products = pd.read_csv(DATA_PATH + "product_master.csv")
promos   = pd.read_csv(DATA_PATH + "promotion_master.csv",   parse_dates=["start_date","end_date"])
stores   = pd.read_csv(DATA_PATH + "store_master.csv")
customers= pd.read_csv(DATA_PATH + "customer_master.csv")
warehouses= pd.read_csv(DATA_PATH + "warehouse_master.csv")
stock_mv = pd.read_csv(DATA_PATH + "stock_movement.csv",     parse_dates=["receive_date","transfer_date"])

tables = {
    "sales": sales, "po": po, "products": products,
    "promos": promos, "stores": stores, "customers": customers,
    "warehouses": warehouses, "stock_movement": stock_mv
}

for name, df in tables.items():
    print(f"{name:<15} → {len(df):>5} rows × {df.shape[1]} cols")

## 1. Data Quality Check

In [ ]:
# ── Null check ───────────────────────────────────────────────
print("=" * 55)
print(f"{'Table':<20} {'Column':<22} {'Null %':>8}")
print("=" * 55)
for name, df in tables.items():
    nulls = df.isnull().mean()
    for col, pct in nulls[nulls > 0].items():
        flag = "⚠️ " if pct > 0.1 else "  "
        print(f"{flag}{name:<18} {col:<22} {pct*100:>7.1f}%")
print("\n✅ No nulls found" if all(df.isnull().sum().sum() == 0 for df in tables.values()) else "")

In [ ]:
# ── Duplicate check ─────────────────────────────────────────
for name, df in tables.items():
    dupes = df.duplicated().sum()
    status = "✅" if dupes == 0 else f"⚠️  {dupes} dupes"
    print(f"{name:<20} → {status}")

In [ ]:
# ── Date range check ────────────────────────────────────────
print(f"Sales period : {sales['datetime'].min().date()} → {sales['datetime'].max().date()}")
print(f"PO period    : {po['po_date'].min().date()} → {po['po_date'].max().date()}")
print(f"Promo period : {promos['start_date'].min().date()} → {promos['end_date'].max().date()}")
print(f"\nUnique products in sales  : {sales['product_id'].nunique()}")
print(f"Unique products in master : {products['product_id'].nunique()}") 
print(f"Coverage                  : {sales['product_id'].nunique()/products['product_id'].nunique()*100:.1f}%")

In [ ]:
# ── Join key validation ─────────────────────────────────────
sales_prods = set(sales['product_id'].unique())
master_prods = set(products['product_id'].unique())
orphan = sales_prods - master_prods
print(f"Products in sales but NOT in master: {len(orphan)} {'✅' if len(orphan)==0 else '⚠️'}")

sales_stores = set(sales['store_id'].unique())
master_stores = set(stores['store_id'].unique())
orphan_s = sales_stores - master_stores
print(f"Stores in sales but NOT in master  : {len(orphan_s)} {'✅' if len(orphan_s)==0 else '⚠️'}")

## 2. Sales Analysis

In [ ]:
# ── Revenue over time (weekly) ───────────────────────────────
sales['revenue'] = sales['price'] * sales['qty']
weekly = sales.resample('W', on='datetime')['revenue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(weekly['datetime'], weekly['revenue'], alpha=0.3, color='steelblue')
ax.plot(weekly['datetime'], weekly['revenue'], color='steelblue', lw=2)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'฿{x:,.0f}'))
ax.set_title('Weekly Revenue — Full Year 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('eda_weekly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total revenue : ฿{sales['revenue'].sum():,.2f}")
print(f"Avg weekly    : ฿{weekly['revenue'].mean():,.2f}")

In [ ]:
# ── Revenue by product category ──────────────────────────────
sales_prod = sales.merge(products[['product_id','product_taxonomies']], on='product_id')
cat_rev = sales_prod.groupby('product_taxonomies')['revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_rev.index, cat_rev.values, color=sns.color_palette('muted', len(cat_rev)))
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'฿{x:,.0f}'))
for bar, val in zip(bars, cat_rev.values):
    ax.text(val + cat_rev.max()*0.01, bar.get_y()+bar.get_height()/2,
            f'฿{val:,.0f}', va='center', fontsize=10)
ax.set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Seasonality: Day of Week + Month ─────────────────────────
sales['dow']   = sales['datetime'].dt.day_name()
sales['month'] = sales['datetime'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_rev = sales.groupby('dow')['revenue'].mean().reindex(dow_order)
axes[0].bar(dow_rev.index, dow_rev.values, color=sns.color_palette('muted', 7))
axes[0].set_title('Avg Daily Revenue by Day of Week')
axes[0].set_xticklabels(dow_order, rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'฿{x:,.0f}'))

# Month
month_rev = sales.groupby('month')['revenue'].sum()
axes[1].bar(month_rev.index, month_rev.values, color=sns.color_palette('flare', 12))
axes[1].set_title('Total Revenue by Month')
axes[1].set_xlabel('Month')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'฿{x:,.0f}'))
axes[1].set_xticks(range(1,13))

plt.tight_layout()
plt.savefig('eda_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Insight: Weekend revenue สูงกว่า weekday เฉลี่ย และมี month-end spike")

## 3. Promotion Effect Analysis

In [ ]:
# ── Promotion vs Non-promotion revenue ───────────────────────
sales['has_promo'] = sales['promotion_id'].notna() & (sales['promotion_id'] != '')

promo_summary = sales.groupby('has_promo').agg(
    avg_qty     = ('qty', 'mean'),
    avg_revenue = ('revenue', 'mean'),
    count       = ('qty', 'count')
).round(2)
promo_summary.index = ['No Promotion', 'With Promotion']
print(promo_summary)
print()

lift_qty = promo_summary.loc['With Promotion','avg_qty'] / promo_summary.loc['No Promotion','avg_qty'] - 1
lift_rev = promo_summary.loc['With Promotion','avg_revenue'] / promo_summary.loc['No Promotion','avg_revenue'] - 1
print(f"Quantity lift  : {lift_qty*100:+.1f}%")
print(f"Revenue lift   : {lift_rev*100:+.1f}%")

In [ ]:
# ── Promotion effect by discount level ───────────────────────
sales_promo = sales[sales['has_promo']].merge(
    promos[['promotion_id','discount']], on='promotion_id', how='left')

disc_effect = sales_promo.groupby('discount').agg(
    avg_qty=('qty','mean'), avg_revenue=('revenue','mean'), count=('qty','count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.bar(disc_effect['discount'].astype(str), disc_effect['avg_qty'],
        color='steelblue', alpha=0.7, label='Avg Qty')
ax2.plot(disc_effect['discount'].astype(str), disc_effect['avg_revenue'],
         color='orange', marker='o', lw=2, label='Avg Revenue')
ax1.set_xlabel('Discount Level')
ax1.set_ylabel('Avg Qty Sold', color='steelblue')
ax2.set_ylabel('Avg Revenue (฿)', color='orange')
ax1.set_title('Promotion Effect by Discount Level', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_promo_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Insight: discount level ที่สูงขึ้น qty เพิ่มแต่ revenue อาจลด → ต้องหา sweet spot")

## 4. Inventory & Supply Chain Analysis

In [ ]:
# ── Lead time distribution ───────────────────────────────────
po['lead_time_days'] = (po['arrival_date'] - po['po_date']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(po['lead_time_days'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Lead Time Distribution (days)')
axes[0].set_xlabel('Days from PO to Arrival')
axes[0].axvline(po['lead_time_days'].mean(), color='red', ls='--', label=f"Mean: {po['lead_time_days'].mean():.1f}d")
axes[0].legend()

# Lead time by warehouse
lt_wh = po.groupby('warehouse_id')['lead_time_days'].agg(['mean','std']).reset_index()
axes[1].bar(lt_wh['warehouse_id'], lt_wh['mean'],
            yerr=lt_wh['std'], capsize=5, color=sns.color_palette('muted', len(lt_wh)))
axes[1].set_title('Avg Lead Time by Warehouse')
axes[1].set_ylabel('Days')

plt.tight_layout()
plt.savefig('eda_lead_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean lead time : {po['lead_time_days'].mean():.1f} days")
print(f"Std lead time  : {po['lead_time_days'].std():.1f} days")
print(f"Max lead time  : {po['lead_time_days'].max()} days ← ⚠️ outlier?")

In [ ]:
# ── Expiry Risk Analysis ─────────────────────────────────────
REFERENCE_DATE = pd.Timestamp('2025-01-01')  # สมมติ ณ วันที่วิเคราะห์
po['days_to_expire'] = (po['expire_date'] - REFERENCE_DATE).dt.days
po['shelf_life_days'] = (po['expire_date'] - po['arrival_date']).dt.days

def expiry_risk(days):
    if days < 0:    return '🔴 Expired'
    elif days < 30: return '🟠 Critical (<30d)'
    elif days < 90: return '🟡 Warning (<90d)'
    else:           return '🟢 Safe'

po['expiry_status'] = po['days_to_expire'].apply(expiry_risk)
risk_counts = po['expiry_status'].value_counts()
print("Expiry Risk Summary:")
print(risk_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'🟢 Safe':'#2ecc71','🟡 Warning (<90d)':'#f39c12',
          '🟠 Critical (<30d)':'#e67e22','🔴 Expired':'#e74c3c'}
for status, count in risk_counts.items():
    ax.bar(status, count, color=colors.get(status, 'gray'))
ax.set_title('PO Expiry Risk Distribution', fontweight='bold')
ax.set_ylabel('Number of POs')
plt.tight_layout()
plt.savefig('eda_expiry_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. EDA Summary — Key Insights

In [ ]:
lines = [
    '=' * 58, '  EDA KEY INSIGHTS SUMMARY', '=' * 58,
    '  1. SEASONALITY',
    '     Weekend revenue สูงกว่า weekday ~40%, Month-end spike ชัดเจน',
    '     Feature: is_weekend, is_month_end',
    '-' * 58,
    '  2. PROMOTION EFFECT',
    '     Transactions with promo: qty สูงกว่า ~60%',
    '     Feature: has_promo_this_week, discount_pct',
    '-' * 58,
    '  3. LEAD TIME: Mean ~8 วัน, std ~3 วัน -> ต้องใช้ M2 lead time model',
    '-' * 58,
    '  4. EXPIRY RISK: มี PO ใกล้ expire -> Expiry engine มี real value',
    '=' * 58,
]
print(chr(10).join(lines))